# Week 3 — 적대적 학습: pix2pix GAN

## 학습 목표
1. L1 손실만 사용할 때 경계가 회색으로 번지는 현상을 확인한다.
2. 두 번째 신경망인 **판별자(Discriminator)** 를 구성하고, hinge 손실을 수치로 이해한다.
3. GAN을 학습한다. 처음부터 학습하는 경로와, W2 체크포인트를 이어받아 미세조정하는 경로를 모두 다룬다.
4. 복원 결과를 원본과 비교하고, GAN이 지표에 미치는 영향을 평가한다.

## 표기 규약
- **G**: 생성자(Generator). 이웃 슬라이스 두 장에서 가운데 슬라이스를 예측한다.
- **D**: 판별자(Discriminator). 슬라이스가 실제 암석 단면인지 점수로 판정한다.
- **λ (lambda_gan)**: 적대적 손실의 가중치. 재구성 손실 대비 GAN 항의 비중을 정한다.
- **k**: 이웃 거리. 측정한 슬라이스 사이의 간격이다.
- **|Δφ|**: 복원 슬라이스와 원본 슬라이스의 공극률 차이(단위 %p). 작을수록 정확하다.
- **SSIM**: 구조 유사도(0~1). 클수록 구조가 유사하다.

> 이번 주 데이터는 **Bentheimer 사암**이다. 구조가 균질해 복원 난이도가 낮은 편이라, 모델의 슬라이스 복원 품질을 관찰하기에 적합하다. 이웃 거리는 **k=2**(측정 슬라이스 사이에 2칸씩 비어 있는 상황)로 설정한다.

> 생성자와 판별자의 관계는 위조범과 감별사의 관계에 비유할 수 있다. 위조범은 진짜에 가까운 위조를 만들려 하고, 감별사는 위조를 가려내려 한다. 두 쪽이 번갈아 성능을 높이면서 위조범의 산출물이 실제에 가까워진다. 이 노트북에서는 위조범이 생성자, 감별사가 판별자에 해당한다.

## 0. 환경 준비

helper 모듈 두 개를 불러온다. `dr_utils` 는 데이터 입출력·보간·평가 지표를, `model_utils` 는 신경망과 학습 루프를 담는다. W3에서 새로 사용하는 함수는 판별자(`PatchDiscriminatorMini`), 손실 함수(`ssim_loss`, `d_hinge_loss`, `g_hinge_loss`), GAN 학습 루프(`train_gan`), 연속 출력 예측(`predict_continuous`)이다. 난수 시드를 고정해 실행마다 결과가 크게 흔들리지 않게 한다.

In [ ]:
import sys
from pathlib import Path

# helper 모듈(dr_utils.py, model_utils.py)은 같은 폴더(다운로드 zip)나 ../helpers(저장소)에 있다.
# 두 위치를 차례로 확인해 먼저 발견되는 쪽을 import 경로에 추가한다.
for _cand in [Path('.'), Path('..') / 'helpers']:
    if (_cand / 'dr_utils.py').exists():
        sys.path.insert(0, str(_cand.resolve())); break

import numpy as np
import matplotlib.pyplot as plt
import torch

# dr_utils: 데이터 I/O · 선형 보간 baseline · 평가 지표(|Δφ|, SSIM) · 색상 팔레트
from dr_utils import (
    load_volume, porosity, predict_linear_k, eval_targets,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
# model_utils: W2 UNet(생성자) + W3에서 추가되는 판별자·손실·GAN 학습 루프
from model_utils import (
    UNetMini, count_parameters, train_quick, evaluate_model, load_ckpt,
    # --- W3에서 새로 쓰는 것 ---
    PatchDiscriminatorMini,   # 조건부 PatchGAN 판별자
    ssim_loss,                # 미분가능 SSIM 손실 (1 − SSIM)
    d_hinge_loss, g_hinge_loss,  # hinge 형태의 판별자/생성자 적대 손실
    train_gan,                # 재구성+적대 손실을 함께 쓰는 GAN 학습 루프
    predict_continuous,       # 이진화 전 연속 출력(0~1) 반환
    save_gan_ckpt, load_gan_ckpt,
)
setup_plot_style()   # 한글 폰트 등록 + 그림 기본 스타일 설정

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)   # 재현성을 위한 시드 고정
print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. W2 복습, 그리고 오늘의 문제

W2에서 만든 UNet은 선형 보간보다 오차가 크게 낮았다. 다만 결과를 자세히 보면 한계가 남는다. 이진화(0/1) 이전의 연속 출력을 보면 pore 경계가 **회색으로 번진다**.

원인은 다음과 같다. 이웃 두 장만으로는 가운데 픽셀이 pore인지 solid인지 확정하기 어려운 위치가 생긴다. 이런 위치에서 L1 손실은 오차 기댓값을 줄이려고 두 값의 **중간값(회색)** 을 출력한다. 이 값은 pore도 solid도 아니며 실제 암석에는 존재하지 않는다. 이를 "평균의 함정(regression to the mean)"이라 부른다.

**이 셀에서 보는 것**: 선형 보간의 기준 지표를 먼저 출력하고, 판별자 없이 L1만으로 학습한 모델(`G_l1`)을 준비한다. `train_gan(lambda_gan=0.0)` 은 적대 손실을 끄고 재구성 손실만 사용하므로, L1 단독 모델을 얻는 경로가 된다.

In [ ]:
# Bentheimer 256^3 이진 부피를 로드한다. data/ 또는 ../data/ 에서 파일을 찾는다.
DATA = next((p for p in [Path('data'), Path('..') / 'data'] if (p / 'Bentheimer_256.bin').exists()), Path('data'))
vol = load_volume(DATA / 'Bentheimer_256.bin')
K = 2   # 이웃 거리 k: 슬라이스 t 를 t±K 에서 예측 (측정 슬라이스 사이 간격 2칸)

# 선형 보간 baseline — 비교의 기준선.
#   predict_linear_k: recon[t] = ( 0.5·(vol[t-K]+vol[t+K]) > 0.5 )
#   eval_targets: 예측 대상 슬라이스에서 |Δφ|(%p)와 SSIM을 평균낸다.
m_lin = eval_targets(predict_linear_k(vol, K), vol, K)
print(f'선형 보간 (k={K})   |Δφ|={m_lin["dphi_pp"]:.2f}%p   SSIM={m_lin["ssim"]:.3f}')

# L1 단독 모델 학습 (판별자 없이 재구성 손실만).
#   lambda_gan=0.0 → 적대 손실 off,  w_ssim=0.0 → SSIM 항도 off → 순수 L1
#   warmup=0 → 어차피 λ=0이므로 몸풀기 구간이 필요 없다
#   d_base=16 → 판별자 폭(여기서는 미사용이나 인자 형식 유지)
print('\nL1 단독 모델 학습 중...')
G_l1, _, _ = train_gan(vol, k=K, preset='fast', lambda_gan=0.0, w_ssim=0.0,
                       epochs=22, warmup=0, d_base=16, device=DEVICE, verbose=True)

### 1.1 회색 번짐 관찰

**무엇을 보는가**: 고정한 patch 하나에서 원본, L1 연속 출력, 그리고 경계를 가로지르는 단면 프로파일을 나란히 본다.

**읽는 법**:
- 왼쪽 두 그림은 같은 위치의 원본과 L1 출력이다. 오렌지 파선이 프로파일을 뽑는 가로줄이다.
- 오른쪽 그래프에서 정답(NAVY)은 pore(1)와 solid(0) 사이를 **계단처럼 급격히** 넘어간다. L1 출력(RED)은 같은 경계를 **0.5 근처로 완만하게** 넘어간다. 이 완만한 구간이 회색 번짐이다.
- 제목의 "회색 %"는 0.2~0.8 범위(불확실 구간)에 있는 픽셀 비율이다. 이 값이 클수록 경계가 흐리다.

In [ ]:
# 고정 위치의 patch 하나를 정해 이후 셀에서 계속 재사용한다(같은 자리로 비교).
z, c0, c1 = 128, 64, 192
before, after, target = vol[z-K, c0:c1, c0:c1], vol[z+K, c0:c1, c0:c1], vol[z, c0:c1, c0:c1]

# predict_continuous: 이진화(0.5 임계) 이전의 연속 출력(0~1)을 반환한다.
cont_l1 = predict_continuous(G_l1, before, after, device=DEVICE)
# 회색(불확실) 픽셀 비율: 값이 0.2~0.8 사이면 pore/solid로 확정되지 않은 픽셀로 본다.
grey = lambda c: ((c > 0.2) & (c < 0.8)).mean()

fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
ax[0].imshow(target, cmap='gray_r', vmin=0, vmax=1); ax[0].set_title('원본 (정답)')
ax[1].imshow(cont_l1, cmap='gray_r', vmin=0, vmax=1); ax[1].set_title(f'L1 출력 · 회색 {grey(cont_l1)*100:.0f}%')
# 경계가 가장 뚜렷한 가로줄을 골라 프로파일로 뽑는다(가로 방향 차분 합이 최대인 행).
row = int(np.argmax(np.abs(np.diff(target, axis=1)).sum(axis=1)))
for a in ax[:2]: a.axhline(row, color=ORANGE, ls='--', lw=1.3); a.set_xticks([]); a.set_yticks([])
ax[2].plot(target[row], color=NAVY, lw=2.4, label='정답 (계단)')
ax[2].plot(cont_l1[row], color=RED, lw=2.2, label='L1 (완만 = 회색)')
ax[2].axhline(0.5, color=GRAY, ls=':', lw=1)   # 이진화 임계선
ax[2].set_xlabel('가로 위치 (픽셀)'); ax[2].set_ylabel('출력값 (0=solid, 1=pore)')
ax[2].legend(fontsize=9); ax[2].set_title('가로 단면 프로파일')
plt.tight_layout(); plt.show()
print('경계에서 값이 0.5 근처로 완만하게 넘어간다. 이 완만한 구간이 회색 번짐이다.')

## 2. 판별자 만들기

GAN의 핵심은 두 번째 신경망인 **판별자 D** 다. 슬라이스를 받아서 "이게 진짜 암석 단면이야?" 하고 점수를 매긴다. 우리 판별자는 세 가지 장치를 쓴다.

- **조건부(conditional)**: 이웃 슬라이스도 같이 넣는다. 그래서 "사실적이면서 이웃과도 맞는가"를 본다.
- **PatchGAN**: 이미지를 조각(patch)으로 나눠서 조각마다 점수를 낸다. 국소적인 경계와 텍스처에 민감해진다.
- **spectral norm**: 판별자의 힘에 상한을 걸어 학습을 안정시킨다.

한 가지 더. 우리 생성자(mini UNet)는 아주 작기 때문에, 판별자를 너무 크게 만들면 판별자가 일방적으로 이겨서 학습이 망가진다. 그래서 판별자 폭을 `d_base=16` 정도로 작게 잡는다.

**무엇을 보는가**: 판별자를 만들어 파라미터 수를 확인하고, 임의 입력을 한 번 통과시켜 출력 형태를 본다. 학습은 하지 않는 형태 점검 셀이다. 출력이 하나의 점수가 아니라 **점수 map**(각 칸이 한 patch의 판정)이라는 점을 확인한다.

In [ ]:
# 판별자 생성: cond_ch=2 → 조건으로 앞·뒤 슬라이스 2채널을 받는다. base=16 → 폭.
D = PatchDiscriminatorMini(cond_ch=2, base=16)
print('D 파라미터:', f'{count_parameters(D):,}')

# 입력 형태 점검: 조건(2채널) + 판정 대상(1채널)을 넣으면 patch별 점수 map이 나온다.
cond = torch.randn(1, 2, 64, 64)   # 조건 = 앞·뒤 슬라이스 (여기서는 임의 값)
y    = torch.rand(1, 1, 64, 64)    # 판정 대상 = 가운데 슬라이스 후보
score = D(cond, y)                 # sigmoid 없는 raw logit map (hinge 손실에서 그대로 사용)
print('입력 조건', tuple(cond.shape), '+ 대상', tuple(y.shape), '-> 점수 map', tuple(score.shape))
print('점수 map의 각 칸이 한 patch에 대한 진짜(+)/가짜(-) 판정이다. 전체를 한 점수로 뭉치지 않는다.')

## 3. 적대적 손실 (hinge)

**판별자 D** 는 진짜에 +1 이상, 가짜에 -1 이하를 주려고 한다.

$$\mathcal{L}_D = \mathbb{E}\,[\max(0,\,1-D(y))] + \mathbb{E}\,[\max(0,\,1+D(G(x)))]$$

**생성자 G** 는 판별자를 속이려고, 즉 점수를 올리려고 한다.

$$\mathcal{L}_G^{\text{adv}} = -\,\mathbb{E}\,[D(G(x))]$$

규칙을 한 문장으로 요약하면, 확실히 맞힌 경우에는 벌점이 0이고 애매한 경우에만 벌점이 생긴다. 아래 셀에서 수치로 확인한다.

In [ ]:
# 판별자가 매긴 점수 예시. 실제 D 출력을 흉내낸 값이다(학습 불필요).
real_score = torch.tensor([0.8, 1.5, -0.2])   # 진짜 슬라이스에 매긴 점수
fake_score = torch.tensor([-0.4, 0.3, -1.5])  # 생성자 출력(가짜)에 매긴 점수

# 판별자 hinge 벌점: relu(1 − D(real)) + relu(1 + D(fake))
#   real 점수가 +1 이상이면 첫 항이 0, fake 점수가 −1 이하이면 둘째 항이 0.
d_pen = torch.relu(1 - real_score).mean() + torch.relu(1 + fake_score).mean()
# 생성자 적대 손실: −D(fake). 판별자 점수를 높일수록(= 잘 속일수록) 작아진다.
g_adv = -fake_score.mean()

print('D 벌점 (진짜는 +1 이상, 가짜는 -1 이하면 0):', round(float(d_pen), 3))
print('G 손실 (-D(fake), 작을수록 잘 속인 것):', round(float(g_adv), 3))
print('real=1.5, fake=-1.5 는 확실히 맞혀 벌점 0. real=0.8, fake=0.3 은 애매해 벌점이 생긴다.')

### 3.1 hinge 벌점 곡선

**무엇을 보는가**: hinge 벌점을 판별자 점수 축 위에서 곡선으로 그린다. 학습이나 데이터가 필요 없는 결정론적 셀이라 즉시 실행할 수 있다.

**읽는 법**:
- 왼쪽은 진짜 슬라이스에 대한 벌점 `relu(1 − s)`이다. 점수 s가 +1 이상이면 벌점이 0으로 평평해지고, +1보다 작아질수록 선형으로 커진다.
- 오른쪽은 가짜 슬라이스에 대한 벌점 `relu(1 + s)`이다. 점수 s가 −1 이하이면 0, −1보다 커질수록 선형으로 커진다.
- 점으로 찍은 예시는 앞 셀의 `real_score`, `fake_score` 값이다. 벌점 0 구간에 놓인 점(진짜 1.5, 가짜 −1.5)은 이미 확실히 판정된 경우다. 이 그림은 강의 슬라이드 23과 같은 개념이다.

In [ ]:
# hinge 벌점의 형태만 확인하는 결정론적 셀 (numpy/matplotlib, 학습 불필요).
s = np.linspace(-3, 3, 400)                 # 판별자 점수 축
pen_real = np.maximum(0.0, 1.0 - s)         # 진짜에 대한 벌점: relu(1 − s)
pen_fake = np.maximum(0.0, 1.0 + s)         # 가짜에 대한 벌점: relu(1 + s)

real_pts = np.array([0.8, 1.5, -0.2])       # 앞 셀의 예시 점수 재사용
fake_pts = np.array([-0.4, 0.3, -1.5])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(s, pen_real, color=NAVY, lw=2.4)
ax[0].scatter(real_pts, np.maximum(0, 1 - real_pts), color=ORANGE, s=70, zorder=3)
ax[0].axvline(1.0, color=GREEN, ls='--', lw=1.2)          # 벌점 0이 시작되는 경계 s=+1
ax[0].text(1.05, 2.2, 's=+1 이상 → 벌점 0', color=GREEN, fontsize=9)
ax[0].set_title('진짜에 대한 벌점  relu(1 − s)')

ax[1].plot(s, pen_fake, color=RED, lw=2.4)
ax[1].scatter(fake_pts, np.maximum(0, 1 + fake_pts), color=ORANGE, s=70, zorder=3)
ax[1].axvline(-1.0, color=GREEN, ls='--', lw=1.2)         # 벌점 0이 시작되는 경계 s=−1
ax[1].text(-2.95, 2.2, 's=−1 이하 → 벌점 0', color=GREEN, fontsize=9)
ax[1].set_title('가짜에 대한 벌점  relu(1 + s)')

for a in ax:
    a.set_xlabel('판별자 점수 s = D(·)'); a.set_ylabel('벌점'); a.grid(alpha=.2)
plt.tight_layout(); plt.show()
print('점(오렌지)은 앞 셀의 예시 점수. 평평한 구간(벌점 0)에 놓인 점은 이미 확실히 판정된 경우다.')

### 3.2 재구성 손실: L1과 SSIM의 차이

생성자 손실은 적대 항 외에 재구성 항(L1, SSIM)을 포함한다. 두 항의 성격이 다르므로 함께 사용한다.

**무엇을 보는가**: 경계가 한 픽셀 어긋난 예측을, 값이 흐려진 예측과 비교한다. 데이터·학습 없이 도는 결정론적 셀이다.

**읽는 법**:
- L1은 픽셀별 절댓값 차이만 본다. 경계가 조금 어긋나거나 값이 흐려지는 두 경우를 비슷한 크기로 벌한다.
- SSIM은 국소 구조(평균·분산·공분산)를 본다. 값이 흐려져 대비가 낮아지면 SSIM 손실이 더 크게 반응한다. 이 때문에 SSIM 항이 경계 대비를 유지하도록 돕는다.

In [ ]:
# L1과 SSIM이 무엇에 반응하는지 보이는 결정론적 예시 (학습 불필요).
N = 64
x = np.arange(N)
gt = np.repeat((x >= N//2).astype(np.float32)[None, :], N, axis=0)   # 급격한 계단(정답)
shifted = np.repeat((x >= N//2 + 1).astype(np.float32)[None, :], N, axis=0)  # 1픽셀 이동(선명 유지)
ramp = np.clip((x - N//2) / 12.0 + 0.5, 0.0, 1.0)                    # 넓게 흐려진(대비 낮은) 경계
blurred = np.repeat(ramp[None, :], N, axis=0).astype(np.float32)

def l1(a, b): return float(np.abs(a - b).mean())
_t = lambda z: torch.from_numpy(z)[None, None]
# ssim_loss = 1 − SSIM (helper). 값이 클수록 국소 구조가 다르다.
ss = lambda a, b: float(ssim_loss(_t(a), _t(b)))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in [(ax[0], gt, '정답 (계단)'), (ax[1], shifted, '예측 A: 경계 1픽셀 이동'),
                 (ax[2], blurred, '예측 B: 경계 흐려짐')]:
    a.imshow(im, cmap='gray_r', vmin=0, vmax=1); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

l1_a, ss_a = l1(shifted, gt), ss(shifted, gt)
l1_b, ss_b = l1(blurred, gt), ss(blurred, gt)
print(f'예측 A(이동)  L1={l1_a:.4f}   SSIM 손실={ss_a:.4f}')
print(f'예측 B(흐림)  L1={l1_b:.4f}   SSIM 손실={ss_b:.4f}')
# 결론은 계산된 값에서 유도한다(하드코딩하지 않음).
worse = 'B(흐림)' if ss_b > ss_a else 'A(이동)'
print(f'SSIM 손실이 더 큰 쪽: {worse}. 대비가 낮아진 흐림을 SSIM이 더 크게 벌해 경계 대비 유지를 돕는다.')

## 4. GAN 학습 (경로 1: 처음부터)

`train_gan` 이 warmup, hinge 손실, spectral norm을 함께 처리한다. 초기 `warmup` epoch 동안은 재구성 손실(L1, SSIM)만 사용하고(λ=0) 이후 판별자를 도입한다. `snapshot` 을 지정하면 지정한 patch의 연속 출력을 주기적으로 저장해 학습 진행을 관찰할 수 있다.

**주요 인자**:
- `lambda_gan=0.1`: 적대 손실 가중치 λ. 재구성 손실 대비 GAN 항의 비중.
- `warmup=8`: 처음 8 epoch은 재구성 손실만으로 학습(λ=0).
- `d_base=16`: 판별자 폭. 생성자가 작으므로 판별자도 작게 잡아 균형을 맞춘다.
- `d_every=1`: 판별자를 매 step 갱신.
- `lambda_decay=0.3`: warmup 이후 진행률에 따라 λ를 점진적으로 낮춘다.
- `snapshot_every=3`: 3 epoch마다 snapshot patch의 출력을 저장.

In [ ]:
# 처음부터(from scratch) GAN 학습. generator 인자를 주지 않으면 새 UNetMini로 시작한다.
G, D, hist = train_gan(vol, k=K, preset='fast',
                       lambda_gan=0.1,      # 적대 손실 가중치 λ
                       w_ssim=0.3,          # 재구성 손실 중 SSIM 항 가중치 (L1 가중치는 기본 1.0)
                       epochs=30, warmup=8, # 총 30 epoch, 앞 8 epoch은 재구성만(λ=0)
                       d_base=16,           # 판별자 폭 (독주 방지용으로 작게)
                       d_every=1,           # 판별자를 매 step 갱신
                       lambda_decay=0.3,    # warmup 이후 λ를 진행률에 따라 감쇠
                       snapshot=(before, after), snapshot_every=3,  # 진행 관찰용 출력 저장
                       device=DEVICE, verbose=True)
# 생성자+판별자 가중치와 meta를 저장한다. W4에서 이 체크포인트를 불러올 수 있다.
save_gan_ckpt(G, D, 'w3_gan_mini.pth', meta={'g_base': 16, 'd_base': 16, 'k': K, 'domain': 'Bentheimer'})
print('\n체크포인트 저장: w3_gan_mini.pth')

### 4.1 학습 곡선 해석

생성자와 판별자가 번갈아 갱신되는 학습이므로, 곡선의 모양으로 학습 상태를 판단한다.

- 왼쪽(재구성 손실 L1·SSIM): GAN 도입 이후에도 값이 계속 낮아지면, 사실감을 더하면서 정확도를 유지한다는 신호다. GAN 도입 직후 소폭 상승했다가 다시 낮아지는 형태도 정상 범위다.
- 오른쪽(판별자 손실 D·생성자 적대 손실 G): **D 손실이 0에 붙지 않고 일정 값을 유지**하면 두 신경망이 균형을 이룬 상태다. D 손실이 0으로 떨어지면 판별자가 일방적으로 우세해 생성자가 더 학습하지 못하는 상태를 의미한다. 초록 음영(0.2~0.8)은 경험적으로 안정적인 D 손실 구간이다.

세로 파선(GAN 시작)을 기준으로 왼쪽은 재구성만 학습한 구간, 오른쪽은 적대 손실이 더해진 구간이다.

In [ ]:
warmup = 8   # 위 학습에서 사용한 warmup 값과 동일하게 맞춘다(파선 위치용)
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))

# 왼쪽: 생성자 재구성 손실. 두 곡선 모두 낮아지는 추세인지 본다.
ax[0].plot(hist['G_l1'], color=NAVY, lw=2, label='L1')
ax[0].plot(hist['G_ssim'], color=ORANGE, lw=2, label='SSIM 손실 (1 − SSIM)')
ax[0].axvline(warmup, color=GRAY, ls='--'); ax[0].text(warmup+0.3, max(hist['G_l1'])*0.85, 'GAN 시작', color=GRAY)
ax[0].set_title('생성자 재구성 손실'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('손실')
ax[0].legend(); ax[0].grid(alpha=.2)

# 오른쪽: 판별자 손실(D)과 생성자 적대 손실(G). 균형 여부를 본다.
ax[1].plot(hist['D_loss'], color=RED, lw=2, label='판별자 D')
ax[1].plot(hist['G_gan'], color=GREEN, lw=2, label='생성자 적대 G')
ax[1].axhline(0, color=GRAY, lw=0.8); ax[1].axvline(warmup, color=GRAY, ls='--')
ax[1].axhspan(0.2, 0.8, color=GREEN, alpha=0.06)   # 경험적 안정 구간
ax[1].set_title('적대적 손실 (D와 G의 균형)'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('손실')
ax[1].legend(); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()

# GAN이 켜진 구간(λ>0)의 D 손실 평균으로 균형 상태를 요약한다.
d_post = np.array(hist['D_loss'])[np.array(hist['lam']) > 0]
print(f'GAN 구간 D 손실 평균 {d_post.mean():.2f} (0.2~0.8 사이면 안정적 균형). 0에 붙지 않았으면 정상이다.')

## 5. 복원 결과와 원본 비교

학습한 GAN 모델로 빠진 슬라이스를 복원해 원본과 나란히 본다.

**읽는 법**: 왼쪽부터 원본, 선형 보간, UNet+GAN 복원이다. 오른쪽 끝은 원본과 GAN 복원의 절대 차이로, `hot` 컬러맵에서 **밝을수록 틀린 픽셀**이다. 이 차이 map이 전반적으로 어두우면 복원이 원본에 가깝다는 뜻이다. `evaluate_model` 은 이진화(0.5 임계) 이후의 복원으로 |Δφ|와 SSIM을 계산한다.

In [ ]:
# evaluate_model: 각 슬라이스 t를 [vol[t-K], vol[t+K]]에서 예측 → 0.5로 이진화 → 지표 계산.
res_gan = evaluate_model(G, vol, k=K, device=DEVICE)
lin_recon = predict_linear_k(vol, K)   # 선형 보간 복원(비교용)
zc = 129   # 복원 대상 슬라이스 하나를 골라 시각화한다
fig, ax = plt.subplots(1, 4, figsize=(15, 4.2))
# 네 패널: 원본 / 선형 / GAN / 차이. 앞 세 개는 gray_r, 마지막은 차이라서 hot.
panels = [('원본 (정답)', vol[zc], NAVY), (f'선형 보간', lin_recon[zc], RED),
          ('UNet + GAN', res_gan['recon'][zc], ORANGE),
          ('|원본 - GAN|', np.abs(vol[zc] - res_gan['recon'][zc]), None)]
for a, (t, im, col) in zip(ax, panels):
    a.imshow(im, cmap='hot' if col is None else 'gray_r', vmin=0, vmax=1)   # 색 한계 고정
    a.set_title(t, color=col if col else 'k', fontsize=13, fontweight='bold'); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print(f'선형 |Δφ|={m_lin["dphi_pp"]:.2f}%p  ->  GAN |Δφ|={res_gan["dphi_pp"]:.2f}%p, SSIM={res_gan["ssim"]:.3f}')
print('균질한 Bentheimer라 복원이 원본에 가깝게 나온다.')

### 5.1 L1 대 GAN: 연속 출력의 선명도

**무엇을 보는가**: 같은 patch에서 L1 출력과 GAN 출력의 연속값을 비교한다. 위쪽 세 그림은 이미지, 아래 그래프는 §1.1과 같은 가로줄의 단면 프로파일이다.

**읽는 법**:
- 위쪽: 제목의 회색 비율이 L1보다 GAN에서 낮으면, GAN 출력의 경계가 더 또렷하다는 뜻이다.
- 아래쪽 프로파일: 정답(NAVY)의 계단에 대해 L1(RED)은 완만하게, GAN(GREEN)은 더 가파르게 넘어간다. 가파를수록 pore/solid 판정이 명확하다. 이것이 GAN이 바꾸는 핵심이다. GAN의 목표는 픽셀 오차를 더 줄이는 것이 아니라 출력을 실제 분포에 가깝게(이진에 가깝게) 만드는 것이다.

In [ ]:
# GAN 모델의 연속 출력을 같은 patch에서 뽑는다(L1과 동일 위치 비교).
cont_gan = predict_continuous(G, before, after, device=DEVICE)

fig = plt.figure(figsize=(13, 7.4))
# 위: 원본 / L1 / GAN 이미지
for i, (im, t, col) in enumerate([(target, '원본', NAVY),
                                  (cont_l1, f'L1 · 회색 {grey(cont_l1)*100:.0f}%', RED),
                                  (cont_gan, f'GAN · 회색 {grey(cont_gan)*100:.0f}%', GREEN)]):
    a = fig.add_subplot(2, 3, i + 1)
    a.imshow(im, cmap='gray_r', vmin=0, vmax=1); a.set_title(t, color=col)
    a.axhline(row, color=ORANGE, ls='--', lw=1.1)   # 아래 프로파일을 뽑는 가로줄 표시
    a.set_xticks([]); a.set_yticks([])
# 아래: 같은 가로줄의 경계 프로파일을 L1과 GAN이 함께 놓고 비교
axp = fig.add_subplot(2, 1, 2)
axp.plot(target[row],   color=NAVY,  lw=2.6, label='정답 (계단)')
axp.plot(cont_l1[row],  color=RED,   lw=2.2, label='L1 (완만)')
axp.plot(cont_gan[row], color=GREEN, lw=2.2, label='GAN (가파름)')
axp.axhline(0.5, color=GRAY, ls=':', lw=1)   # 이진화 임계선
axp.set_xlabel('가로 위치 (픽셀)'); axp.set_ylabel('출력값 (0=solid, 1=pore)')
axp.set_title('경계 단면 프로파일 비교'); axp.legend(fontsize=9); axp.grid(alpha=.2)
plt.tight_layout(); plt.show()
print(f'회색(불확실) 비율: L1 {grey(cont_l1)*100:.0f}% -> GAN {grey(cont_gan)*100:.0f}%. '
      f'GAN 쪽 경계가 더 가파르고 또렷하다.')

## 6. 평가: 선형 vs L1 vs GAN

세 방법을 같은 조건에서 비교한다. 공정한 비교를 위해 GAN 없는 모델(`G_fair`)도 GAN 모델과 동일한 epoch·설정으로 학습한다. 차이는 오직 `lambda_gan`(적대 손실 on/off)뿐이다.

In [ ]:
# 공정 비교 조건: G_fair는 lambda_gan=0.0(적대 손실 off), 나머지 설정은 §4의 GAN 학습과 동일.
#   → GAN 유무만 다르게 두어 λ의 순수 효과를 본다.
G_fair, _, _ = train_gan(vol, k=K, preset='fast', lambda_gan=0.0, w_ssim=0.3,
                         epochs=30, warmup=8, d_base=16, device=DEVICE, verbose=False)
res_fair = evaluate_model(G_fair, vol, k=K, device=DEVICE)
# 세 방법을 표로 정렬해 |Δφ|와 SSIM을 나란히 출력한다.
print(f'{"방법":<16}{"|Δφ|(%p)":>10}{"SSIM":>9}')
print(f'{"선형 보간":<15}{m_lin["dphi_pp"]:>10.2f}{m_lin["ssim"]:>9.3f}')
print(f'{"UNet (GAN 없음)":<13}{res_fair["dphi_pp"]:>10.2f}{res_fair["ssim"]:>9.3f}')
print(f'{"UNet + GAN":<15}{res_gan["dphi_pp"]:>10.2f}{res_gan["ssim"]:>9.3f}')

> **관찰**: 선형 보간에서 딥러닝으로 넘어가면 오차가 크게 줄어든다(약 11.75%p에서 1.2%p 아래로). GAN을 켜면 이 데이터에서는 |Δφ|가 한 번 더 낮아졌다. 다만 이 차이는 크지 않고, 학습의 무작위성에 따라 비슷하거나 소폭 반대로 나올 수도 있다. 여러 번 반복 학습하면 분포를 파악할 수 있다.
>
> 분명한 것은 **GAN이 연속 출력을 더 선명하게** 만든다는 점이다(위에서 본 회색 비율). GAN의 목표는 픽셀 오차를 더 줄이는 것이 아니라 출력을 실제 분포에 가깝게 만드는 것이므로, 그 효과가 선명도에서 먼저 나타난다. 큰 모델에서는 투과율 같은 물성 지표에서 이득이 더 뚜렷하다(강의 슬라이드의 LBM 결과). 픽셀 지표 하나로는 이 차이를 충분히 포착하지 못하므로, S2나 투과율 같은 물리 지표도 함께 확인한다(W5).

## 7. 경로 2: W2 체크포인트 이어받아 미세조정

처음부터 학습하는 대신, W2에서 만든 UNet을 생성자 초기값으로 이어받을 수도 있다. 이미 기본기가 있어서 warmup을 짧게 줄일 수 있다. (여기서는 W2 모델을 즉석에서 하나 만들어 보여준다. 실제로는 W2에서 저장한 `unet_mini_*.pth` 를 `load_ckpt` 로 불러오면 된다.)

In [ ]:
# 1) W2 방식의 L1 UNet을 하나 준비한다(실전에서는 load_ckpt로 저장본을 불러온다).
G0, _ = train_quick(vol, k=K, preset='fast', device=DEVICE, verbose=False)
print('W2 UNet 준비 완료. 이어서 GAN 미세조정...')
# 2) generator=G0 로 넘기면 새 모델 대신 이 가중치에서 이어 학습한다(fine-tune).
#    이미 재구성 기본기가 있으므로 warmup을 2로 짧게 준다.
G_ft, D_ft, _ = train_gan(vol, k=K, generator=G0,
                          lambda_gan=0.1, w_ssim=0.3,
                          epochs=15, warmup=2,   # from-scratch(30/8)보다 짧게
                          d_base=16, device=DEVICE, verbose=True)
cont_ft = predict_continuous(G_ft, before, after, device=DEVICE)
print(f'\n미세조정 결과 회색 비율: {grey(cont_ft)*100:.0f}%. W2 이어받기는 warmup을 짧게 줄일 수 있다.')

## 8. 심화: λ와 흐림-환각 절충

적대 손실 가중치 **λ** 가 절충을 결정한다. λ가 너무 작으면 L1에 가까워 경계가 흐리고, 너무 크면 이웃과 맞지 않는 구조를 만들어내는 환각(hallucination)이 생긴다. λ를 여러 값으로 바꿔 선명도(회색 비율)와 |Δφ|가 어떻게 변하는지 관찰한다.

**읽는 법**: λ가 커질수록 회색 비율은 낮아지는(선명해지는) 경향이 있으나, |Δφ|는 어느 지점 이후 다시 나빠질 수 있다. 두 지표가 함께 좋은 지점이 실전에서 선택하는 λ다.

In [ ]:
# λ를 바꿔가며 선명도(회색 비율)와 정확도(|Δφ|, SSIM)의 변화를 관찰한다.
for lam in [0.0, 0.1, 0.3]:
    Gi, _, _ = train_gan(vol, k=K, preset='fast', lambda_gan=lam, w_ssim=0.3,
                         epochs=26, warmup=6, d_base=16, device=DEVICE, verbose=False)
    ci = predict_continuous(Gi, before, after, device=DEVICE)   # 연속 출력 → 회색 비율
    ri = evaluate_model(Gi, vol, k=K, device=DEVICE)            # 이진화 후 지표
    print(f'λ={lam:<4}  회색 {grey(ci)*100:4.0f}%   |Δφ| {ri["dphi_pp"]:.2f}%p   SSIM {ri["ssim"]:.3f}')
print('\nGAN 학습은 λ에 민감하다. λ를 바꾸면 결과가 상당히 변한다.')
print('λ를 크게 넣으면 |Δφ|가 오히려 나빠지기 쉽다(위 표).')
print('따라서 실전에서는 λ를 작게(0.1 근처) 두어 재구성 손실이 학습을 주도하게 한다.')

## 9. 정리 및 다음 주

**핵심 정리**
- L1 손실은 불확실한 위치에서 중간값(회색)을 출력한다. 이것이 평균의 함정이다.
- GAN은 생성자와 판별자를 경쟁시켜, 회색 대신 실제에 가까운 구조를 생성하도록 유도한다.
- 조건부 입력, PatchGAN, spectral norm, hinge 손실, 작은 λ가 실전 구성이다.
- 판별자가 지나치게 우세하면(D 손실이 0에 붙으면) 학습이 정체된다. 판별자 폭을 작게(`d_base=16`) 잡아 균형을 맞췄다.
- GAN은 출력을 더 선명하게 만들었고, 이 데이터에서는 |Δφ|도 함께 개선되었다. 다만 픽셀 지표만으로는 GAN의 이득을 충분히 포착하지 못하므로, 큰 모델에서는 투과율 같은 물성 지표로 확인한다(W5).

**다음 주 (W4)**: 다른 아키텍처(Transformer 기반, 3D)를 같은 문제에 적용해 이번 주 GAN과 비교한다. 학습한 체크포인트(`w3_gan_mini.pth`)를 보관한다.

**탐구 과제**(핸드아웃 §5): GAN 유무 비교, λ sweep, warmup·spectral norm 실험, 지표는 유사하나 구조가 다른 사례 탐색.